# HDFS 3.2.2 historical baseline on QEMU

Run the existing NameNode and DataNode roles in a single Ubuntu guest. This is a pseudo-distributed HDFS baseline (one replica), not HA, YARN, or a migration test. Keep the same guest disk and NameNode/DataNode directories throughout all phases.


In [ ]:
import hashlib
import json
import os
from pathlib import Path
import re
import shlex
import subprocess
import time

root = Path.cwd()
state = Path(os.environ['CI_STATE'])
artifacts = Path(os.environ['CI_ARTIFACTS'])
inventory = state / 'inventory.yml'
variables = root / 'ci/hdfs-vars.yml'
assert inventory.is_file(), inventory
resolved = json.loads(subprocess.check_output(['ansible-inventory', '-i', str(inventory), '--list'], text=True))
for group in ['hadoop_all', 'hadoop_namenode', 'hadoop_namenode_primary', 'hadoop_slavenode']:
    assert resolved.get(group, {}).get('hosts') == ['notebook-ci'], resolved

def run(*args):
    print('+', ' '.join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), check=True, timeout=1200)

def playbook(path):
    run('ansible-playbook', '-i', inventory, '-e', '@' + str(variables), root / path)

def remote(*args):
    results = state / 'command-results'
    run('ansible', '-i', inventory, 'hadoop_all', '-b', '--become-user', 'hdfs',
        '-m', 'command', '-a', shlex.join(map(str, args)), '--tree', results)
    result = json.loads((results / 'notebook-ci').read_text())
    assert result.get('rc') == 0, result
    return result['stdout']

def verify_data(label):
    # Wait for DataNode registration and safemode exit after each restart.
    remote('timeout', '120', 'hdfs', 'dfsadmin', '-safemode', 'wait')
    report = remote('hdfs', 'dfsadmin', '-report')
    assert 'Live datanodes (1)' in report, report
    remote('hdfs', 'dfs', '-get', '-f', '/ci/persistent.bin', '/tmp/hdfs-ci-readback.bin')
    actual = remote('sha256sum', '/tmp/hdfs-ci-readback.bin').split()[0]
    assert actual == expected_sha256, (label, expected_sha256, actual)
    fsck = remote('hdfs', 'fsck', '/ci/persistent.bin', '-files', '-blocks')
    assert 'HEALTHY' in fsck and 'CORRUPT' not in fsck, fsck
    assert re.search(r'3 block\(s\)', fsck), fsck
    print(f'{label}: SHA-256 matched; 3 blocks HEALTHY', flush=True)


## Bind, install Hadoop, and initialize the empty namespace once

In [ ]:
run('ansible', '-i', inventory, 'all', '-m', 'ping')
playbook('playbooks/install_namenode.yml')
playbook('playbooks/install_slavenode.yml')
version = remote('hdfs', 'version')
assert version.splitlines()[0] == 'Hadoop 3.2.2', version
print('Baseline version verified: Hadoop 3.2.2')
playbook('playbooks/format_namenode.yml')
playbook('playbooks/start_namenode.yml')
playbook('playbooks/start_datanode.yml')
playbook('ci/enable-hdfs.yml')


## Create a deterministic three-block data fixture and read it back

In [ ]:
fixture = state / 'persistent.bin'
fixture.write_bytes(bytes(range(256)) * 12288)  # 3 MiB; 1 MiB HDFS blocks.
expected_sha256 = hashlib.sha256(fixture.read_bytes()).hexdigest()
(artifacts / 'expected-sha256.txt').write_text(expected_sha256 + '\n')
run('ansible', '-i', inventory, 'hadoop_all', '-b', '-m', 'copy',
    '-a', f'src={fixture} dest=/tmp/hdfs-ci-input.bin mode=0644')
remote('timeout', '120', 'hdfs', 'dfsadmin', '-safemode', 'wait')
remote('hdfs', 'dfs', '-mkdir', '-p', '/ci')
remote('hdfs', 'dfs', '-Ddfs.blocksize=1048576', '-put', '/tmp/hdfs-ci-input.bin', '/ci/persistent.bin')
verify_data('Initial read')


## Reject a second format and prove the existing namespace is unchanged

In [ ]:
before = remote('sha256sum', '/var/lib/hadoop-hdfs/namenode/current/VERSION')
log_path = artifacts / 'expected-format-refusal.log'
with log_path.open('w') as log:
    result = subprocess.run(['ansible-playbook', '-i', str(inventory), '-e', '@' + str(variables),
        str(root / 'ci/check-format-guard.yml')], stdout=log, stderr=subprocess.STDOUT, timeout=120)
print(log_path.read_text())
assert result.returncode != 0, 'Formatting existing NameNode unexpectedly succeeded'
assert 'Refusing to format existing NameNode storage' in log_path.read_text()
assert remote('sha256sum', '/var/lib/hadoop-hdfs/namenode/current/VERSION') == before
verify_data('After rejected reformat')


## Stop/start both HDFS services without formatting or recreating disks

In [ ]:
playbook('playbooks/stop_datanode.yml')
playbook('playbooks/stop_namenode.yml')
playbook('playbooks/start_namenode.yml')
playbook('playbooks/start_datanode.yml')
verify_data('After service restart')


## Reboot the same guest, then read the original HDFS data

In [ ]:
run('ansible', '-i', inventory, 'all', '-b', '-m', 'reboot', '-a', 'reboot_timeout=180')
verify_data('After VM reboot')
remote('hdfs', 'dfs', '-touchz', '/ci/written-after-reboot')
remote('hdfs', 'dfs', '-test', '-e', '/ci/written-after-reboot')
print('Post-reboot writes verified')


## Next: upgrade on the same data directories

A future scenario can stop this baseline, upgrade Hadoop without formatting the namespace, then verify the recorded SHA-256 and new writes. This baseline does not establish upgrade compatibility, HA/failover, or replication durability.
